In [ ]:
pip install -r ../requirements.txt


# Medical Diagnosis Using Bayesian Networks — Heart Disease Prediction

## PGM project overview

This notebook walks through all **three pillars** of Probabilistic Graphical Models:

| Pillar | Implementation |
|--------|----------------|
| **Representation** | Manual Structure + learned DAGs over symptoms and HeartDisease |
| **Learning** | MLE/Bayesian CPTs; Hill Climb structure learning |
| **Inference** | Variable Elimination & Belief Propagation |

**Query answered:** P(Heart Disease | observed symptoms and risk factors)


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.config import MANUAL_STRUCTURE_EDGES, TARGET, STATE_LABELS
from src.data import load_discretized_dataset, train_test_split_data, dataset_summary, records_to_evidence
from src.model import build_manual_structure, representation_summary
from src.learning import learn_manual_structure_bn, learn_structure_and_parameters, get_learning_summary
from src.inference import predict_disease, disease_probability
from src.evaluation import evaluate_model, results_to_dataframe, benchmark_inference_methods
from src.config import ProjectConfig
from src.visualization import plot_dag

OUT = ROOT / "outputs" / "notebook"
OUT.mkdir(parents=True, exist_ok=True)
print("Root:", ROOT)


## PART 1 — Data & preprocessing (UCI multi-source)


In [ ]:
df = load_discretized_dataset()
summary = dataset_summary(df)
print("Samples:", summary["n_samples"])
print("By source:", summary["by_source"])
print("Prevalence:", f"{summary['prevalence']:.1%}")
df.head()


## PART 2 — Representation (manual structure Bayesian Network)


In [ ]:
meta = build_manual_structure()
print(meta["description"])
print("Edges:", meta["edges"])


## PART 3 — Learning (CPTs + structure)


In [ ]:
train_df, test_df = train_test_split_data(df)
config = ProjectConfig()
manual_lr = learn_manual_structure_bn(train_df)
learned_lr = learn_structure_and_parameters(train_df, config)
print(get_learning_summary(manual_lr))
print(get_learning_summary(learned_lr))
plot_dag(manual_lr.model, OUT / "manual_structure_dag.png", "Manual Structure BN")
plot_dag(learned_lr.model, OUT / "learned_dag.png", "Learned BN")


## PART 4 — Inference (VE vs BP)


In [ ]:
row = test_df.iloc[0]
evidence = records_to_evidence(row)
for method in ("ve", "bp"):
    trace = predict_disease(manual_lr.model, evidence, method=method)
    print(method.upper(), "P(Yes)=", f"{disease_probability(trace):.3f}", f"({trace.elapsed_ms:.1f} ms)")
    print("  Notes:", trace.notes[0])


## PART 5 — Evaluation


In [ ]:
results = []
for lr, name in [(manual_lr, "Manual Structure"), (learned_lr, "Learned")]:
    for method in ("ve", "bp"):
        results.append(evaluate_model(lr.model, test_df, name, method=method))
metrics = results_to_dataframe(results)
metrics


## PART 6 — Summary


| Part | PGM pillar | Deliverable |
|------|------------|-------------|
| 1 | Data | UCI heart disease merged & discretized |
| 2 | Representation | Manual Structure DAG |
| 3 | Learning | CPTs + Hill Climb structure |
| 4 | Inference | VE & BP queries |
| 5 | Evaluation | Accuracy, precision, recall, F1, ROC-AUC |

**Live demo:** `streamlit run app/streamlit_app.py`
